# 16 - Model-Based Reinforcement Learning (Dyna-Q)

## Learning Objectives
1. Understand how Dyna-Q interleaves real experience and simulated planning
2. Implement Dyna-Q on a numpy GridWorld and compare K=0 vs K=5 vs K=20
3. Build and train a neural world model (sklearn MLP) for planning
4. Analyze model error compounding over short vs long rollout horizons


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print('Packages loaded: numpy', np.__version__)
print('Environment: pure-numpy GridWorld (no gym required)')


## Level 1: Basic Dyna-Q on GridWorld

GridWorld: 5x5 grid, agent starts at (0,0), goal at (4,4). Actions: 0=up, 1=down, 2=left, 3=right. Reward: +1 at goal, -0.01 per step, walls block movement. Compare K=0 (pure Q-learning) vs K=5 vs K=20 planning steps per real step.


In [ ]:
# GridWorld environment (no gym)
class GridWorld:
    def __init__(self, rows=5, cols=5):
        self.rows = rows
        self.cols = cols
        self.goal = (rows - 1, cols - 1)
        self.n_states = rows * cols
        self.n_actions = 4  # up, down, left, right
        self.deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def reset(self):
        self.pos = (0, 0)
        return self._state()

    def _state(self):
        return self.pos[0] * self.cols + self.pos[1]

    def step(self, action):
        r, c = self.pos
        dr, dc = self.deltas[action]
        nr, nc = r + dr, c + dc
        nr = max(0, min(self.rows - 1, nr))  # clip to grid
        nc = max(0, min(self.cols - 1, nc))
        self.pos = (nr, nc)
        done = self.pos == self.goal
        reward = 1.0 if done else -0.01
        return self._state(), reward, done


def run_dyna_q(K=0, n_episodes=300, alpha=0.1, gamma=0.95, eps=0.1):
    env = GridWorld()
    Q = np.zeros((env.n_states, env.n_actions))
    model = {}  # model[(s, a)] = (r, s')
    past_sa = []  # list of observed (s, a) pairs
    episode_rewards = []
    episode_steps = []

    for ep in range(n_episodes):
        s = env.reset()
        total_reward = 0
        steps = 0
        done = False

        while not done and steps < 200:
            # Epsilon-greedy action selection
            if np.random.random() < eps:
                a = np.random.randint(env.n_actions)
            else:
                a = np.argmax(Q[s])

            s2, r, done = env.step(a)
            total_reward += r
            steps += 1

            # Direct Q-learning update (real experience)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])

            # Model update: store deterministic transition
            model[(s, a)] = (r, s2)
            if (s, a) not in past_sa:
                past_sa.append((s, a))

            # Planning: K simulated updates from model
            for _ in range(K):
                if not past_sa:
                    break
                idx = np.random.randint(len(past_sa))
                sp, ap = past_sa[idx]
                rp, s2p = model[(sp, ap)]
                Q[sp, ap] += alpha * (rp + gamma * np.max(Q[s2p]) - Q[sp, ap])

            s = s2

        episode_rewards.append(total_reward)
        episode_steps.append(steps)

    return episode_rewards, episode_steps, Q


print('Training Dyna-Q K=0 (pure Q-learning)...')
r0, steps0, Q0 = run_dyna_q(K=0, n_episodes=300)
print(f'  Final 50-ep mean reward: {np.mean(r0[-50:]):.3f}, steps: {np.mean(steps0[-50:]):.1f}')

print('Training Dyna-Q K=5...')
r5, steps5, Q5 = run_dyna_q(K=5, n_episodes=300)
print(f'  Final 50-ep mean reward: {np.mean(r5[-50:]):.3f}, steps: {np.mean(steps5[-50:]):.1f}')

print('Training Dyna-Q K=20...')
r20, steps20, Q20 = run_dyna_q(K=20, n_episodes=300)
print(f'  Final 50-ep mean reward: {np.mean(r20[-50:]):.3f}, steps: {np.mean(steps20[-50:]):.1f}')


## Level 2: Neural World Model

Train a neural dynamics model f_theta(s, a) -> (r, s') using sklearn MLPRegressor. Use it for planning: generate K simulated transitions, update Q-table. Track model prediction error (MSE) over training.


In [ ]:
# Neural World Model (sklearn MLP)
class NeuralWorldModel:
    # MLP that predicts (next_state, reward) from (state, action)

    def __init__(self, n_states, n_actions):
        self.n_states = n_states
        self.n_actions = n_actions
        # Separate models for reward and next-state prediction
        self.reward_model = MLPRegressor(
            hidden_layer_sizes=(32, 32), activation='relu',
            max_iter=5, warm_start=True, random_state=42
        )
        self.next_state_model = MLPRegressor(
            hidden_layer_sizes=(32, 32), activation='relu',
            max_iter=5, warm_start=True, random_state=42
        )
        self.buffer_X = []
        self.buffer_r = []
        self.buffer_s2 = []
        self.trained = False

    def _encode(self, s, a):
        # One-hot encode state and action into feature vector
        x = np.zeros(self.n_states + self.n_actions)
        x[s] = 1.0
        x[self.n_states + a] = 1.0
        return x

    def update(self, s, a, r, s2):
        # Add transition to buffer and retrain if enough data
        x = self._encode(s, a)
        self.buffer_X.append(x)
        self.buffer_r.append(r)
        self.buffer_s2.append(s2)
        # Retrain every 10 transitions (warm_start=True keeps weights)
        if len(self.buffer_X) >= 20 and len(self.buffer_X) % 10 == 0:
            X = np.array(self.buffer_X)
            self.reward_model.fit(X, np.array(self.buffer_r))
            self.next_state_model.fit(X, np.array(self.buffer_s2))
            self.trained = True

    def predict(self, s, a):
        # Predict (reward, next_state) for state-action pair
        if not self.trained:
            return 0.0, s
        x = self._encode(s, a).reshape(1, -1)
        r_pred = self.reward_model.predict(x)[0]
        s2_pred = int(np.round(np.clip(
            self.next_state_model.predict(x)[0], 0, self.n_states - 1
        )))
        return r_pred, s2_pred

    def prediction_mse(self):
        # Compute MSE on training buffer
        if not self.trained or len(self.buffer_X) < 20:
            return float('nan')
        X = np.array(self.buffer_X)
        r_pred = self.reward_model.predict(X)
        s2_pred = self.next_state_model.predict(X)
        r_mse = np.mean((r_pred - np.array(self.buffer_r)) ** 2)
        s2_mse = np.mean((s2_pred - np.array(self.buffer_s2)) ** 2)
        return r_mse + s2_mse


def run_neural_dyna(K=5, n_episodes=300, alpha=0.1, gamma=0.95, eps=0.1):
    env = GridWorld()
    Q = np.zeros((env.n_states, env.n_actions))
    world_model = NeuralWorldModel(env.n_states, env.n_actions)
    past_sa = []
    model_errors = []
    episode_rewards = []

    for ep in range(n_episodes):
        s = env.reset()
        total_reward = 0
        done = False
        steps = 0

        while not done and steps < 200:
            if np.random.random() < eps:
                a = np.random.randint(env.n_actions)
            else:
                a = np.argmax(Q[s])

            s2, r, done = env.step(a)
            total_reward += r
            steps += 1

            # Real Q-update
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])

            # Neural model update
            world_model.update(s, a, r, s2)
            if (s, a) not in past_sa:
                past_sa.append((s, a))

            # Planning using neural model
            for _ in range(K):
                if not past_sa or not world_model.trained:
                    break
                sp, ap = past_sa[np.random.randint(len(past_sa))]
                rp, s2p = world_model.predict(sp, ap)
                Q[sp, ap] += alpha * (rp + gamma * np.max(Q[s2p]) - Q[sp, ap])

            s = s2

        episode_rewards.append(total_reward)
        model_errors.append(world_model.prediction_mse())

    return episode_rewards, model_errors


print('Training neural Dyna-Q (K=5)...')
r_neural, errors = run_neural_dyna(K=5, n_episodes=300)
print(f'Final 50-ep mean reward: {np.mean(r_neural[-50:]):.3f}')
valid_errors = [e for e in errors if not (isinstance(e, float) and e != e)]
if valid_errors:
    print(f'Final model MSE: {np.mean(valid_errors[-50:]):.4f}')


## Real-World Example 1: Sample Efficiency Comparison

Compare real-environment steps needed to reach a performance threshold for model-free (K=0), Dyna-Q (K=5, K=20), and neural model-based approaches. Run multiple seeds for stable estimates.


In [ ]:
def smooth(arr, window=20):
    return np.convolve(arr, np.ones(window)/window, mode='valid')

def steps_to_solve(rewards, threshold=-0.5, window=20):
    smoothed = smooth(rewards, window)
    above = np.where(smoothed > threshold)[0]
    return above[0] + window if len(above) > 0 else len(rewards)

n_seeds = 5
results = {k: [] for k in [0, 5, 20, 'neural']}

for seed in range(n_seeds):
    np.random.seed(seed)
    r0_, _, _ = run_dyna_q(K=0, n_episodes=400)
    np.random.seed(seed)
    r5_, _, _ = run_dyna_q(K=5, n_episodes=400)
    np.random.seed(seed)
    r20_, _, _ = run_dyna_q(K=20, n_episodes=400)
    np.random.seed(seed)
    r_n_, _ = run_neural_dyna(K=5, n_episodes=400)
    results[0].append(r0_)
    results[5].append(r5_)
    results[20].append(r20_)
    results['neural'].append(r_n_)

colors = {0: '#2c7bb6', 5: '#d7191c', 20: '#1a9641', 'neural': '#fd8d3c'}
labels = {0: 'K=0 (Q-learning)', 5: 'K=5 (Dyna-Q)', 20: 'K=20 (Dyna-Q)', 'neural': 'Neural model K=5'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for k, color in colors.items():
    all_r = np.array(results[k])
    mean_r = np.mean(all_r, axis=0)
    std_r = np.std(all_r, axis=0)
    smoothed_mean = smooth(mean_r)
    ep_range = np.arange(len(smoothed_mean))
    ax.plot(ep_range, smoothed_mean, color=color, label=labels[k], linewidth=2)
    sm_std = smooth(std_r)
    ax.fill_between(ep_range, smoothed_mean - sm_std,
                    smoothed_mean + sm_std, alpha=0.2, color=color)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward (smoothed)', fontsize=12)
ax.set_title('Sample Efficiency: Dyna-Q vs Neural vs Q-learning', fontsize=13)
ax.legend(fontsize=10)
ax.set_facecolor('#f8f8f8')
ax.grid(True, alpha=0.4)

ax2 = axes[1]
solve_eps = {k: [steps_to_solve(results[k][s]) for s in range(n_seeds)] for k in results}
method_names = ['K=0', 'K=5', 'K=20', 'Neural K=5']
means = [np.mean(solve_eps[k]) for k in [0, 5, 20, 'neural']]
stds = [np.std(solve_eps[k]) for k in [0, 5, 20, 'neural']]
bar_colors = [colors[k] for k in [0, 5, 20, 'neural']]
bars = ax2.bar(method_names, means, yerr=stds, capsize=6,
               color=bar_colors, alpha=0.8, edgecolor='black')
ax2.set_ylabel('Episodes to Solve (lower=better)', fontsize=12)
ax2.set_title('Sample Efficiency: Episodes to Threshold', fontsize=13)
ax2.set_facecolor('#f8f8f8')
ax2.grid(True, alpha=0.4, axis='y')
for bar, mean_val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{mean_val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/mbrl_sample_efficiency.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Speedup K=20 vs K=0: {means[0]/max(means[2],1):.1f}x fewer episodes')


## Real-World Example 2: Model Error Propagation

Demonstrate how imperfect world model errors compound over long rollout horizons. Short rollouts (1-3 steps) from model stay near real experience. Long rollouts (10-20 steps) accumulate compounding prediction error.


In [ ]:
def collect_random_data(n_transitions=500):
    env = GridWorld()
    data = []
    s = env.reset()
    for _ in range(n_transitions):
        a = np.random.randint(env.n_actions)
        s2, r, done = env.step(a)
        data.append((s, a, r, s2))
        s = env.reset() if done else s2
    return data


def build_tabular_model(data):
    model = {}
    for s, a, r, s2 in data:
        model[(s, a)] = (r, s2)
    return model


def simulate_rollout(model, Q, start_state, n_actions, rollout_len):
    s = start_state
    total_reward = 0.0
    for _ in range(rollout_len):
        a = np.argmax(Q[s]) if np.random.random() > 0.1 else np.random.randint(n_actions)
        if (s, a) in model:
            r, s2 = model[(s, a)]
        else:
            r, s2 = 0.0, s  # OOD: no model prediction
        total_reward += r
        s = s2
    return total_reward


np.random.seed(42)
data_limited = collect_random_data(n_transitions=300)  # partial coverage
model_limited = build_tabular_model(data_limited)
coverage = len(model_limited) / (25 * 4) * 100
print(f'Model coverage: {len(model_limited)} / {25*4} state-action pairs ({coverage:.1f}%)')

_, _, Q_test = run_dyna_q(K=10, n_episodes=200)

rollout_lengths = [1, 2, 3, 5, 10, 15, 20]
ood_rates = []
cumulative_errors = []
env_for_eval = GridWorld()

for rl in rollout_lengths:
    ood_count = 0
    total_steps_counted = 0
    reward_errors = []
    for trial in range(200):
        s_start = np.random.randint(env_for_eval.n_states)
        sim_reward = simulate_rollout(model_limited, Q_test, s_start, env_for_eval.n_actions, rl)

        # Real rollout
        env_for_eval.pos = (s_start // env_for_eval.cols, s_start % env_for_eval.cols)
        real_reward = 0.0
        s = s_start
        for _ in range(rl):
            a = (np.argmax(Q_test[s]) if np.random.random() > 0.1
                 else np.random.randint(env_for_eval.n_actions))
            s2, r_real, done = env_for_eval.step(a)
            real_reward += r_real
            if (s, a) not in model_limited:
                ood_count += 1
            total_steps_counted += 1
            s = s2
            if done:
                break
        reward_errors.append(abs(sim_reward - real_reward))

    ood_rates.append(ood_count / max(total_steps_counted, 1) * 100)
    cumulative_errors.append(np.mean(reward_errors))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(rollout_lengths, ood_rates, 'o-', color='#d7191c', linewidth=2, markersize=8)
ax1.set_xlabel('Rollout Length (steps)', fontsize=12)
ax1.set_ylabel('OOD Step Rate (%)', fontsize=12)
ax1.set_title('OOD Rate vs Rollout Length', fontsize=12)
ax1.axhline(y=50, color='gray', linestyle='--', label='50% OOD threshold')
ax1.legend()
ax1.set_facecolor('#f8f8f8')
ax1.grid(True, alpha=0.4)

ax2.plot(rollout_lengths, cumulative_errors, 's-', color='#2c7bb6', linewidth=2, markersize=8)
ax2.set_xlabel('Rollout Length (steps)', fontsize=12)
ax2.set_ylabel('Mean |Sim - Real| Reward Error', fontsize=12)
ax2.set_title('Cumulative Prediction Error vs Rollout Length', fontsize=12)
ax2.set_facecolor('#f8f8f8')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('/tmp/mbrl_error_compounding.png', dpi=100, bbox_inches='tight')
plt.show()
print('Key insight: error compounds with rollout length')
ratio = cumulative_errors[-1] / max(cumulative_errors[0], 1e-6)
print(f'Error ratio rollout=20 vs rollout=1: {ratio:.1f}x')


## Real-World Example 3: Latent World Model

Encode GridWorld state into a compact latent representation (PCA). Train world model in latent space - this mirrors how DreamerV2/RSSM works at scale. Evaluate 1-step prediction accuracy of the latent dynamics model.


In [ ]:
from sklearn.decomposition import PCA


def build_latent_model(data, latent_dim=4, n_states=25, n_actions=4):
    states = np.array([d[0] for d in data])
    actions = np.array([d[1] for d in data])
    next_states = np.array([d[3] for d in data])
    rewards = np.array([d[2] for d in data])

    # Encoder: PCA on one-hot states -> latent_dim dimensions
    state_ohe = np.eye(n_states)[states]
    next_state_ohe = np.eye(n_states)[next_states]
    pca = PCA(n_components=latent_dim, random_state=42)
    pca.fit(state_ohe)
    z_current = pca.transform(state_ohe)   # shape (N, latent_dim)
    z_next = pca.transform(next_state_ohe)  # shape (N, latent_dim)

    action_ohe = np.eye(n_actions)[actions]

    # Latent dynamics model: (z_t, a_t) -> z_{t+1}
    dynamics_input = np.concatenate([z_current, action_ohe], axis=1)
    dynamics_model = MLPRegressor(
        hidden_layer_sizes=(32, 32), activation='relu',
        max_iter=200, random_state=42
    )
    dynamics_model.fit(dynamics_input, z_next)

    # Reward model in latent space
    reward_model = MLPRegressor(
        hidden_layer_sizes=(16,), activation='relu',
        max_iter=200, random_state=42
    )
    reward_model.fit(z_current, rewards)

    # Decoder: z -> state index (for evaluation)
    decoder = MLPRegressor(
        hidden_layer_sizes=(16,), activation='relu',
        max_iter=200, random_state=42
    )
    decoder.fit(z_current, states)

    return pca, dynamics_model, reward_model, decoder


def evaluate_latent_model(pca, dynamics_model, reward_model, decoder,
                          data, n_states=25, n_actions=4):
    states = np.array([d[0] for d in data])
    actions = np.array([d[1] for d in data])
    next_states = np.array([d[3] for d in data])
    rewards = np.array([d[2] for d in data])

    state_ohe = np.eye(n_states)[states]
    action_ohe = np.eye(n_actions)[actions]
    z_current = pca.transform(state_ohe)
    dynamics_input = np.concatenate([z_current, action_ohe], axis=1)

    z_next_pred = dynamics_model.predict(dynamics_input)
    next_state_pred = np.round(
        np.clip(decoder.predict(z_next_pred), 0, n_states - 1)
    ).astype(int)
    reward_pred = reward_model.predict(z_current)

    state_acc = np.mean(next_state_pred == next_states)
    reward_mse = np.mean((reward_pred - rewards) ** 2)
    return state_acc, reward_mse


np.random.seed(42)
data_large = collect_random_data(n_transitions=1000)
pca_enc, dyn_model, rew_model, dec = build_latent_model(data_large, latent_dim=4)

np.random.seed(99)
data_eval = collect_random_data(n_transitions=200)
acc, rmse = evaluate_latent_model(pca_enc, dyn_model, rew_model, dec, data_eval)
print(f'Latent model (4D) evaluation:')
print(f'  Next-state prediction accuracy: {acc:.3f}')
print(f'  Reward prediction MSE: {rmse:.5f}')

# Test multiple latent dims
for ldim in [2, 4, 8, 12]:
    pca_t, dyn_t, rew_t, dec_t = build_latent_model(data_large, latent_dim=ldim)
    a_t, r_t = evaluate_latent_model(pca_t, dyn_t, rew_t, dec_t, data_eval)
    explained = np.sum(pca_t.explained_variance_ratio_) * 100
    msg = f'  latent_dim={ldim}: acc={a_t:.3f}, reward_mse={r_t:.5f}, explained_var={explained:.1f}%'
    print(msg)

# Visualize latent space (first 2 PCA dims)
state_ohe_all = np.eye(25)
z_all = pca_enc.transform(state_ohe_all)
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(z_all[:, 0], z_all[:, 1], c=np.arange(25),
                cmap='viridis', s=120, edgecolors='black', linewidths=0.5)
for i in range(25):
    ax.annotate(f'({i//5},{i%5})', (z_all[i, 0], z_all[i, 1]),
                textcoords='offset points', xytext=(4, 4), fontsize=7)
plt.colorbar(sc, ax=ax, label='State index')
ax.set_xlabel('Latent dim 1 (PCA)', fontsize=12)
ax.set_ylabel('Latent dim 2 (PCA)', fontsize=12)
ax.set_title('GridWorld States in Latent Space (first 2 dims)', fontsize=12)
ax.set_facecolor('#f8f8f8')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/mbrl_latent_space.png', dpi=100, bbox_inches='tight')
plt.show()


## Comparison: Dyna-Q (K=0, K=5, K=20) vs Neural Model

Sample efficiency curves and final performance summary across all methods.


In [ ]:
np.random.seed(0)
r0_comp, _, _ = run_dyna_q(K=0, n_episodes=350)
np.random.seed(0)
r5_comp, _, _ = run_dyna_q(K=5, n_episodes=350)
np.random.seed(0)
r20_comp, _, _ = run_dyna_q(K=20, n_episodes=350)
np.random.seed(0)
r_n_comp, _ = run_neural_dyna(K=5, n_episodes=350)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
w = 20

ax = axes[0]
for label, r_arr, color in [
    ('K=0 (Q-learning)', r0_comp, '#2c7bb6'),
    ('K=5 (Dyna-Q)', r5_comp, '#d7191c'),
    ('K=20 (Dyna-Q)', r20_comp, '#1a9641'),
    ('Neural K=5', r_n_comp, '#fd8d3c'),
]:
    sm = smooth(r_arr, window=w)
    ax.plot(np.arange(len(sm)), sm, label=label, linewidth=2)

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Smoothed Reward', fontsize=12)
ax.set_title('Learning Curves: All Methods (window=20 smoothing)', fontsize=12)
ax.legend(fontsize=10)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_facecolor('#f8f8f8')
ax.grid(True, alpha=0.4)

ax2 = axes[1]
final_means = [np.mean(r[-50:]) for r in [r0_comp, r5_comp, r20_comp, r_n_comp]]
method_names_comp = ['K=0\n(Q-learn)', 'K=5\n(Dyna-Q)', 'K=20\n(Dyna-Q)', 'Neural\nK=5']
bar_colors_comp = ['#2c7bb6', '#d7191c', '#1a9641', '#fd8d3c']
bars = ax2.bar(method_names_comp, final_means, color=bar_colors_comp,
               alpha=0.85, edgecolor='black')
ax2.set_ylabel('Mean Reward (last 50 eps)', fontsize=12)
ax2.set_title('Final Performance Comparison', fontsize=12)
ax2.set_facecolor('#f8f8f8')
ax2.grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars, final_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/mbrl_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Comparison Summary:')
print(f'{"Method":<22} {"Final Reward":<15}')
print('-' * 38)
for name, fm in zip(['K=0 (baseline)', 'K=5 Dyna-Q', 'K=20 Dyna-Q', 'Neural K=5'], final_means):
    print(f'{name:<22} {fm:<15.3f}')


## Key Takeaways

**Core idea:** Dyna-Q uses a learned model M(s,a) -> (r, s') to generate K simulated transitions after each real step, improving sample efficiency proportional to K without additional real interactions.

**Variants and when to use:**

| Method | Use when | Trade-off |
|--------|----------|-----------|
| K=0 (Q-learning) | Cheap environment | No model overhead |
| Dyna-Q K=5-20 | Tabular, accurate model | K x compute per real step |
| Neural MBPO | High-dim, data-scarce | Model training cost + error |
| Latent RSSM | Image/complex states | Encoder complexity |

**Common failure modes:**
- K too high + inaccurate model -> worse than K=0 (compounding error)
- Planning from unvisited (s,a) -> OOD Q-values
- No model retraining -> distribution shift between policy and model

**Related concepts:**
- [05-q-learning](../notebooks/05-q-learning.ipynb) - Base Q-learning
- [20-offline-rl](./20-offline-rl.ipynb) - Also avoids OOD actions


## Exercises

1. **Tune K**: In `run_dyna_q`, try K=50. Does performance improve or degrade vs K=20?
2. **Model accuracy vs K**: Collect only 100 random transitions before training. Does K=20 still outperform K=0?
3. **Stochastic environment**: Add 20% random action noise to GridWorld. How does this affect the deterministic model accuracy?
4. **Latent dimension sweep**: Retrain latent model with latent_dim=2 vs 8. How does state reconstruction accuracy change?
